# Hover supplement research — offline walkthrough

All claims and evidence below are fictional. Explicit test models make no API calls. This notebook runs extraction, sampling, post-hoc analysis and prediction without the app or database. The generated reports are local and git-ignored.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
backend = next(p for p in [here, *here.parents] if (p / "ad_hoc").is_dir())
if str(backend) not in sys.path:
    sys.path.insert(0, str(backend))

In [ ]:
import pandas as pd

from ad_hoc.hover_supplements import (
    ResearchConfig,
    build_feature_table,
    extract_batch,
    feature_dictionary,
    run_analysis,
    stratified_sample,
    write_report,
)
from ad_hoc.hover_supplements.synthetic import synthetic_bundles
from app.core.llm import LLMModelAPI, LLMModelConfig

output = backend / "data" / "hover_research" / "notebook"

## 1. Full structured population and a weighted audit sample

The outcome is positive approved additional dollars. A denied request can still have supplement activity. Each populated stratum is sampled; inclusion probabilities are retained.

In [ ]:
bundles = synthetic_bundles(240)
population = pd.DataFrame([b.metadata() for b in bundles])
counts = population.groupby(["hover", population.supplement_approved_amount.gt(0)]).size()
sizes = {tuple(key): min(12, int(n)) for key, n in counts.items()}
sample = stratified_sample(population, sizes)
selected_ids = set(sample.claim_id)
selected = [b for b in bundles if b.claim_id in selected_ids]
sample[["claim_id", "hover", "inclusion_probability", "sampling_weight"]].head()

## 2. Two isolated extraction passes

Use notebook `await`, not `asyncio.run`. Checkpoints can resume successful extraction. For a real study, supply your own bundles, omit `demo=True`, and use the existing backend LLM configuration.

In [ ]:
test_config = LLMModelConfig(model_name="test", api=LLMModelAPI.TEST)
results = await extract_batch(
    selected,
    test_config,
    checkpoint_dir=output / "checkpoints",
    concurrency=4,
    demo=True,
)
assert all(r.status == "success" for r in results)
features = build_feature_table(results, sample, drop_text=False)
features[
    [
        "claim_id",
        "baseline_status",
        "supplement_status",
        "supplement_incidence",
        "supplement_mechanism__history__total_count",
        "supplement_mechanism__primary_mechanism",
        "supplement_mechanism__primary_outcome",
        "supplement_mechanism__secondary_mechanism",
        "supplement_mechanism__remaining_outcome",
        "supplement_mechanism__remaining_avoidability",
    ]
].head()

## 3. Feature inventory and evidence QA

The 115 schema-v2 fields use Annotated pandas metadata. One row contains baseline, whole-history counts/indicators, and linked main/remaining classifications. Unknown values remain missing; skipped and failed passes are distinct. Inspect citations before trusting a classification.

In [ ]:
dictionary = feature_dictionary()
assert len(dictionary) == 115
display(dictionary)
example = next(r for r in results if r.supplement.status == "success")
example.supplement.model_dump()

## 4. Research models and post-hoc decomposition

Full-population structured results and weighted audited results remain separate. This example explicitly treats roof involvement as a pre-existing damage characteristic; real studies must justify their selected covariates. Forty bootstrap replicates keep the walkthrough short; the research default is 500. Sparse audited samples may return unsupported fits or insufficient intervals, which are retained as results.

In [ ]:
config = ResearchConfig(
    tier="enriched",
    covariates=["baseline__property_complexity__roof_involved"],
    pretreatment_rationale={
        "baseline__property_complexity__roof_involved": (
            "For these fictional fixtures, roof involvement was generated before Hover assignment."
        )
    },
    bootstrap_replicates=40,
    trees=30,
    min_segment_size=5,
)
report = run_analysis(population, results=results, audited_metadata=sample, config=config)
display(report["tables"]["population_scorecard"])
display(report["tables"]["mechanism_rates"])
display(report["tables"]["classifications"])
report["population_decomposition"]

In [ ]:
display(report["tables"]["audit_coverage"])
for name in ["structured_model", "selected_model"]:
    model = report[name]
    print(name, model["status"], model.get("reason", model.get("interval_status")))
    display(model.get("estimates", pd.DataFrame()))
prediction = report["predictive"]
print("Predictive status:", prediction["status"], prediction.get("reason", ""))
if prediction["status"] == "success":
    display(prediction["metrics"])
    display(prediction["segment_comparisons"])

## 5. Export and rerun without extraction

Local CSV/JSON/Markdown/HTML reports include coverage, model failures, feature dictionaries, representative evidence, and predictive diagnostics. Mechanism flags overlap. Avoidability-associated dollars are not recoverable savings. Adjusted differences are associations, not established causal effects.

In [ ]:
write_report(report, output / "report")
output.mkdir(parents=True, exist_ok=True)
population.to_csv(output / "population.csv", index=False)
sample.to_json(output / "sample.jsonl", orient="records", lines=True)
(output / "extractions.jsonl").write_text(
    "\n".join(r.model_dump_json() for r in results), encoding="utf-8"
)
print(output / "report" / "summary.md")

## 6. Business questions, predictive explanations, and evidence

The offline report organizes findings around business questions, with explicit units and denominators. Install the optional `explainability` group for held-out SHAP. Importance explains predictions; it does not establish Hover's causal effect. Read held-out accuracy and coverage first. The expected-dollar bridge reconciles separate probability/severity contributions and is not joint SHAP.


In [ ]:
from ad_hoc.hover_supplements import (
    build_business_figures,
    business_question_answers,
    claim_explanation_figure,
    render_business_report,
)

display(business_question_answers(report))
figures = build_business_figures(report)
figures["dollar_decomposition"].show()
figures["initial_quality"].show()

In [ ]:
explanations = report["predictive"].get("explanations", {})
print("Explanation status:", explanations.get("status"), explanations.get("reason", ""))
if explanations.get("status") == "success":
    display(explanations["global_importance"])
    claim_id = explanations["predictions"].claim_id.iloc[0]
    claim_explanation_figure(explanations, claim_id).show()
    claim_explanation_figure(explanations, claim_id, output="expected_dollars").show()
    evidence = explanations["evidence"]
    display(evidence[evidence.claim_id.eq(claim_id)])

### Re-render saved results without extraction or model fitting

The full report includes model diagnostics, signed contribution charts, claim examples selected by risk and prediction error, evidence provenance, unknowns, and unsupported comparisons. All charts work offline.


In [ ]:
import json

saved = json.loads((output / "report" / "research.json").read_text(encoding="utf-8"))
report_path = render_business_report(saved, output / "rendered_again")
print(report_path)
assert report_path.exists()